# Stage 2: Task-Specific AQG Training (Adapter-Based)

**Version**: 3.0 (Adapter Layers - No LoRA)

Notebook ini melatih model IndoNanoT5 untuk task AQG menggunakan **Adapter Layers** (bukan LoRA).

**Key Differences from v2:**
- ✅ Adapter Layers (3.6% trainable params) vs LoRA (0.36%)
- ✅ 99.6% performance dari full fine-tuning
- ✅ No inference latency overhead
- ✅ More stable untuk small datasets
- ✅ 8 epochs training (vs 3 epochs di v2)

**Configuration:**
- Adapter dimension: d=64 (reduction_factor=12)
- Trainable parameters: ~8.9M (3.6% dari 248M)
- Learning rate: 1e-4
- Batch size: 4 + gradient accumulation 2 (effective=8)
- Epochs: 8
- Memory: ~12-14GB (safe untuk T4)

**Expected Results:**
- Training loss: 39 → 2-5
- BLEU-4: 0.005 → 0.20-0.28
- ROUGE-L: 0.0 → 0.25-0.35
- Training time: ~6-8 jam pada T4 GPU

**Reference:** Based on Houlsby et al. (2019) - Parameter-Efficient Transfer Learning for NLP

## 1. Setup Environment

In [ ]:
# Install dependencies
!pip install -q adapter-transformers
!pip install -q transformers datasets accelerate
!pip install -q evaluate rouge_score bert_score
print('✓ Dependencies installed')

In [ ]:
# Cek versi semua library yang terinstall
import importlib
import sys, torch, platform

libs = [
    "adapters",
    "transformers",
    "datasets",
    "accelerate",
    "evaluate",
    "torch",
    "tokenizers",
    "rouge_score",
    "bert_score",
]

print(f"Python:  {sys.version}")
print(f"OS:      {platform.system()}")
print(f"Torch:   {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}\n")

print("=== Library Versions ===")
for lib in libs:
    try:
        mod = importlib.import_module(lib.replace("-", "_"))
        version = getattr(mod, "__version__", "unknown")
        print(f"  {lib:<20} {version}")
    except ImportError:
        print(f"  {lib:<20} NOT INSTALLED")

if torch.cuda.is_available():
    print(f"\n  {'cuda version':<20} {torch.version.cuda}")
    print(f"  {'gpu name':<20} {torch.cuda.get_device_name(0)}")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✓ Google Drive mounted')

In [ ]:
# Setup paths and extract source code
import os, sys, zipfile, shutil

DRIVE_ROOT = '/content/drive/MyDrive/dataset_aqg'
sys.path.insert(0, '/content')

# Extract src if not exists
if not os.path.exists('/content/src'):
    shutil.copy(f'{DRIVE_ROOT}/src_finetuned.zip', '/content/src_finetuned.zip')
    with zipfile.ZipFile('/content/src_finetuned.zip', 'r') as z:
        z.extractall('/content/')
    print('✓ src extracted')
else:
    print('✓ src already exists')

print(f'✓ DRIVE_ROOT: {DRIVE_ROOT}')
print(f'✓ sys.path[0]: {sys.path[0]}')

In [ ]:
# Verify GPU availability
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU not available! Go to Runtime > Change runtime type > T4 GPU')

print(f'✓ GPU: {torch.cuda.get_device_name(0)}')
print(f'✓ Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Load Model with Adapter Layers

**Using**: `src.finetuned.utils.adapter_loader` (modular approach!)

In [ ]:
from src.finetuned.utils.adapter_loader import load_model_with_adapter, print_adapter_info

# Load model with adapter layers
model, tokenizer = load_model_with_adapter(
    model_name='LazarusNLP/IndoNanoT5-base',
    adapter_name='mcq_generation',
    adapter_config='pfeiffer',
    reduction_factor=12,  # d=64
    device='cuda'
)

# Print detailed info
trainable, total = print_adapter_info(model, tokenizer)

## 3. Load Dataset

**Using**: `src.finetuned.data.dataset_loader` (reusable module)

In [ ]:
from src.finetuned.data.dataset_loader import DatasetLoader

loader = DatasetLoader()
TASK_DIR = '/content/dataset_aqg/dataset-task-spesifc/'

# Copy dataset from Drive if needed
if not os.path.exists(TASK_DIR + 'train.jsonl'):
    drive_task = f'{DRIVE_ROOT}/dataset-task-spesifc'
    os.makedirs(TASK_DIR, exist_ok=True)
    for f in ['train.jsonl', 'validation.jsonl', 'test.jsonl']:
        shutil.copy(f'{drive_task}/{f}', f'{TASK_DIR}{f}')
    print('✓ Dataset copied from Drive')

# Load datasets
train_dataset = loader.load_dataset(TASK_DIR, split='train')
val_dataset = loader.load_dataset(TASK_DIR, split='validation')
test_dataset = loader.load_dataset(TASK_DIR, split='test')

print(f'\nDataset loaded:')
print(f'  Train: {len(train_dataset)} samples')
print(f'  Val:   {len(val_dataset)} samples')
print(f'  Test:  {len(test_dataset)} samples')

# Validate and preview
validation_results = loader.validate_dataset(train_dataset)

sample = train_dataset[0]
print('\n=== Sample Entry ===')
print(f"Input: {sample['input'][:100]}...")
output_field = 'target' if 'target' in sample else 'output'
print(f"Output: {sample[output_field][:100]}...")
print(f'\n✓ Dataset ready (supports both v2 and v3 formats)')

## 4. Baseline Evaluation

**Using**: `src.finetuned.evaluation` modules

In [ ]:
from src.finetuned.evaluation.metrics_calculator import MetricsCalculator
from src.finetuned.evaluation.model_evaluator import ModelEvaluator

metrics_calc = MetricsCalculator()
evaluator = ModelEvaluator(
    model=model,
    tokenizer=tokenizer,
    metrics_calculator=metrics_calc
)

print('Computing baseline metrics (10 samples)...')
baseline_metrics = evaluator.evaluate_on_test_set(
    test_dataset=val_dataset,
    num_beams=4,
    include_bertscore=False,
    max_samples=10
)

print(f"\nBaseline Metrics:")
print(f"  BLEU-4:  {baseline_metrics.get('bleu_4', 0):.4f}")
print(f"  ROUGE-L: {baseline_metrics.get('rouge_l', 0):.4f}")

## 5. Configure Training

**Using**: `src.finetuned.training.adapter_trainer` (modular approach!)

In [ ]:
from src.finetuned.training.adapter_trainer import AdapterTrainer

CHECKPOINT_DIR = '/content/drive/MyDrive/dataset_aqg/checkpoints/adapter_v3'

# Initialize trainer
trainer = AdapterTrainer(
    model=model,
    tokenizer=tokenizer,
    metrics_calculator=metrics_calc,
    output_dir=CHECKPOINT_DIR,
    max_length=512
)

# Setup training configuration
training_args = trainer.setup_training(
    num_train_epochs=8,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_steps=50,
    weight_decay=0.01
)

print('\n✓ Trainer configured')
print(f'  Checkpoints will be saved to: {CHECKPOINT_DIR}')

## 6. Start Training

**⚠️ This will take 6-8 hours on T4 GPU**

In [ ]:
import time

start_time = time.time()

# Train (all logic in adapter_trainer.py)
results = trainer.train(
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    training_args=training_args,
    early_stopping_patience=2
)

elapsed = (time.time() - start_time) / 3600
print(f'\n✓ Training completed in {elapsed:.2f} hours')
print(f'  Final training loss: {results["training_loss"]:.4f}')

## 7. Save Adapter & Visualize

In [ ]:
# Save adapter weights
adapter_save_path = trainer.save_adapter(
    adapter_name='mcq_generation',
    save_config={
        "model_name": "LazarusNLP/IndoNanoT5-base",
        "adapter_config": "pfeiffer",
        "reduction_factor": 12,
        "trainable_params": trainable,
        "total_params": total,
        "num_train_epochs": 8,
        "learning_rate": 1e-4,
        "training_time_hours": elapsed
    }
)

# Plot training curves
trainer.plot_training_curves(
    save_path=f'{CHECKPOINT_DIR}/training_curves.png'
)

## 8. Final Evaluation

In [ ]:
# Re-initialize evaluator with trained model
evaluator_final = ModelEvaluator(
    model=model,
    tokenizer=tokenizer,
    metrics_calculator=metrics_calc
)

print('Running comprehensive evaluation on test set...')
final_metrics = evaluator_final.evaluate_on_test_set(
    test_dataset=test_dataset,
    num_beams=4,
    include_bertscore=True,
    max_samples=None
)

print('\n=== Evaluation Results ===')
for key, value in final_metrics.items():
    print(f'{key}: {value:.4f}')

## 9. Generate Sample Outputs

In [ ]:
EVAL_DIR = '/content/drive/MyDrive/dataset_aqg/evaluation_results_v3'

samples = evaluator_final.generate_samples(
    test_dataset=test_dataset,
    num_samples=20,
    num_beams=4,
    save_path=f'{EVAL_DIR}/sample_outputs.json'
)

print(f'✓ {len(samples)} samples generated')

# Preview first 3 samples
if samples:
    print('\n=== Sample Outputs ===')
    for i, sample in enumerate(samples[:3], 1):
        print(f"\n--- Sample {i} ---")
        print(f"Input: {sample['input'][:80]}...")
        print(f"Generated: {sample['generated'][:80]}...")

## 10. Final Summary

In [ ]:
import json
from pathlib import Path

# Compare with baseline
comparison = evaluator_final.compare_with_baseline(
    finetuned_metrics=final_metrics,
    baseline_metrics=baseline_metrics
)

# Save evaluation report
Path(EVAL_DIR).mkdir(parents=True, exist_ok=True)
report = {
    'version': '3.0 (Adapter Layers)',
    'baseline_metrics': baseline_metrics,
    'final_metrics': final_metrics,
    'comparison': comparison,
    'training_time_hours': elapsed,
    'adapter_path': adapter_save_path,
    'config': {
        'adapter_config': 'pfeiffer',
        'reduction_factor': 12,
        'learning_rate': 1e-4,
        'batch_size': 8,
        'epochs': 8
    }
}

with open(f'{EVAL_DIR}/evaluation_report.json', 'w') as f:
    json.dump(report, f, indent=2, default=str)

# Print summary
print('\n' + '='*60)
print('ADAPTER-BASED AQG TRAINING SUMMARY')
print('='*60)
print(f'Method: Adapter Layers (d=64)')
print(f'Training Time: {elapsed:.2f} hours')
print(f'Trainable: {100*trainable/total:.2f}%')
print(f'\nMetrics Comparison:')
print(f"  BLEU-4:  {baseline_metrics.get('bleu_4',0):.4f} → {final_metrics.get('bleu_4',0):.4f}")
print(f"  ROUGE-L: {baseline_metrics.get('rouge_l',0):.4f} → {final_metrics.get('rouge_l',0):.4f}")

bleu_improvement = comparison.get('bleu_4_improvement_pct', 0)
print(f'\nBLEU-4 Improvement: {bleu_improvement:+.1f}%')

if final_metrics.get('bleu_4', 0) >= 0.20:
    print('\n✓ SUCCESS: BLEU-4 target achieved (>= 0.20)')
else:
    print(f"\n⚠ BLEU-4 = {final_metrics.get('bleu_4',0):.4f} (target: >= 0.20)")
    print('  Consider: more epochs or adjust hyperparameters')

print('\n✓ Fine-tuning pipeline complete!')
print(f'  Adapter: {adapter_save_path}')
print(f'  Report: {EVAL_DIR}/evaluation_report.json')
print(f'  Samples: {EVAL_DIR}/sample_outputs.json')

print('\n' + '='*60)
print('HOW TO LOAD TRAINED ADAPTER')
print('='*60)
print('from adapters import AutoAdapterModel')
print('from transformers import AutoTokenizer')
print('')
print('model = AutoAdapterModel.from_pretrained("LazarusNLP/IndoNanoT5-base")')
print('tokenizer = AutoTokenizer.from_pretrained("LazarusNLP/IndoNanoT5-base")')
print(f'model.load_adapter("{adapter_save_path}")')
print('model.set_active_adapters("mcq_generation")')
print('')
print('# Generate')
print('inputs = tokenizer("generate_mcq: [CONTEXT]", return_tensors="pt")')
print('outputs = model.generate(**inputs, max_length=512, num_beams=4)')
print('print(tokenizer.decode(outputs[0], skip_special_tokens=True))')